In [137]:
import pymupdf4llm
import re
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.documents import Document
from langchain_chroma import Chroma

In [138]:
file_path = "Attention.pdf"
markdown = pymupdf4llm.to_markdown(file_path)


=== Document parser messages ===
Using RapidOCR for OCR processing.
OCR on page.number=2/3.
OCR on page.number=3/4.


In [139]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
    ("####", "Header 4")
]

splitter = MarkdownHeaderTextSplitter(headers_to_split_on = headers_to_split_on)
markdown_chunks = splitter.split_text(markdown)

In [140]:
def additional_recursive_chunker(markdown_chunks : list[Document], chunk_size = 600, chunk_overlap = 100):

    good_chunk = []
    chunk_count = 0
    recursive_text_splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size, 
                                                             chunk_overlap = chunk_overlap, 
                                                             separators=["\n\n", "\n", ". ", " ", ""])
    
    for chunk in markdown_chunks:
        if(len(chunk.page_content) > chunk_size):
            meta_data = chunk.metadata
            chunks = recursive_text_splitter.split_documents([chunk])
            for i in chunks:
                chunk_count = chunk_count + 1
                i.metadata["chunk_id"] = chunk_count
                good_chunk.append(i)
            chunk_count = 0
        else:
            good_chunk.append(chunk)
    return good_chunk

In [141]:
# this is the final chunk and metadata which is to be stores and embedded
final_chunks = additional_recursive_chunker(markdown_chunks)

In [142]:
embedding_model = OllamaEmbeddings(model="llama3.2")

In [143]:
# embed each of the query and then store them in the pgvector
vector_store = Chroma(
    collection_name= "my_docs",
    embedding_function=embedding_model,
    persist_directory="./store"
)

In [65]:
ids = vector_store.add_documents(
    documents = final_chunks,)

In [129]:
len(ids)

83

In [144]:
filter_condition = {}
search_kwargs = {}

In [145]:
if len(filter_condition) == 0:
    search_kwargs = {
        'k' : 3,
        "fetch_k" : 10,
        "lambda_mult" : 0.7,
    }
else:
    search_kwargs = {
        'k' : 3,
        "fetch_k" : 10,
        "lambda_mult" : 0.7,
        "filter" : {
            **filter_condition
        }
    },

In [146]:
search_kwargs

{'k': 3, 'fetch_k': 10, 'lambda_mult': 0.7}

In [147]:
retriver = vector_store.as_retriever(
    search_type = "mmr",
    search_kwargs = search_kwargs
)

In [148]:
ans =retriver.invoke("How does an attention mechanism work?")

In [149]:
ans

[Document(id='35e55b56-3c2d-4fb3-aded-3ecb083fe0c5', metadata={'Header 2': '**1 Introduction**', 'Header 1': '**Attention Is All You Need**', 'chunk_id': 7}, page_content='Attention mechanisms have become an integral part of compelling sequence modeling and transduction models in various tasks, allowing modeling of dependencies without regard to their distance in the input or output sequences [2, 16]. In all but a few cases [22], however, such attention mechanisms are used in conjunction with a recurrent network.'),
 Document(id='28bcbc81-a251-4209-a789-b6de8865539c', metadata={'Header 2': '**3 Model Architecture**', 'Header 1': '**Attention Is All You Need**', 'Header 3': 'Ouput Probabilities', 'Header 4': '**3.2.3 Applications of Attention in our Model**', 'chunk_id': 2}, page_content='- The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the encoder. Each

In [150]:
for i in ans:
    print(i.metadata)

{'Header 2': '**1 Introduction**', 'Header 1': '**Attention Is All You Need**', 'chunk_id': 7}
{'Header 2': '**3 Model Architecture**', 'Header 1': '**Attention Is All You Need**', 'Header 3': 'Ouput Probabilities', 'Header 4': '**3.2.3 Applications of Attention in our Model**', 'chunk_id': 2}
{'Header 3': 'Ouput Probabilities', 'Header 4': '**3.2 Attention**', 'Header 2': '**3 Model Architecture**', 'Header 1': '**Attention Is All You Need**'}


### Evaluation of the rag system without hybrid search and reranking

### RAG with hybrid search

In [ ]:
# we need to perform the retrieval on the basis of the semantic and the keyword and the fuse then perform rrf on them to find the best chunk
# semantic search is the normal search for this


### BM25 as the keyword search

In [136]:
# keyword search is done using the bm25 algorithm

### Combining the chunks from dense and sparse search

In [29]:
# rrf

In [30]:
# reranker

### Generation of the answer from the llm

In [32]:
# generation

### Evaluate the chunks and answer after the changes in the pipeline

In [33]:
# evaluation criterias